In [6]:
import anthropic
import json
import os

from dotenv import load_dotenv
from ollama import chat

load_dotenv()

anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

def get_claude_completion(model, prompt: str, system_prompt="", prefill="", temp=1.0, max_tokens=8192):
    message = anthropic_client.messages.create(
        model=model,
        max_tokens=max_tokens,
        temperature=temp,
        system=system_prompt,
        messages=[
          {"role": "user", "content": prompt},
          {"role": "assistant", "content": prefill}
        ]
    )
    return message.content[0].text

In [47]:
######################################## INPUT VARIABLES ########################################

# AGENT_TASK = "Which undergraduate college did Dzung Pham (PhD student at UMass) go to? What's the ranking of that college according to US News?"
# AGENT_TASK = "What is the best college for Computer Science in 2025? Analyze this using different ranking websites such as USNews and CSRanking.org. Make sure to consider factors like location, employment opportunities, etc."
AGENT_TASK = "Find all person names at https://dzungvpham.github.io/cv/cv_2025.pdf"

######################################## PROMPT ELEMENTS ########################################

##### Prompt element 1: `user` role
# Make sure that your Messages API call always starts with a `user` role in the messages array.
# The get_claude_completion() function as defined above will automatically do this for you.

##### Prompt element 2: Task context
# Give Claude context about the role it should take on or what goals and overarching tasks you want it to undertake with the prompt.
# It's best to put context early in the body of the prompt.
TASK_CONTEXT = "You are an AI tasked with assessing the difficulty of a given task for a local AI agent powered by a large language model (LLM). Your goal is to determine the most cost-efficient local LLM size to complete the task. Accurate difficulty estimation is crucial for optimal LLM selection."
##### Prompt element 3: Tone context
# If important to the interaction, tell Claude what tone it should use.
# This element may not be necessary depending on the task.
TONE_CONTEXT = ""

##### Prompt element 4: Detailed task description and rules
# Expand on the specific tasks you want Claude to do, as well as any rules that Claude might have to follow.
# This is also where you can give Claude an "out" if it doesn't have an answer or doesn't know.
# It's ideal to show this description and rules to a friend to make sure it is laid out logically and that any ambiguous words are clearly defined.
TASK_DESCRIPTION = f"""First, you will be presented with the task that the agent needs to solve. Read it carefully:

<agent_task>
{AGENT_TASK}
</agent_task>

The agent will alternate between calling the LLMs to reason about the task's current state and executing tools. The agent can use the following tools: web_search, visit_webpage.

To assess the difficulty of the task, follow these steps:
1. Consider how the agent will plan and break down the task into concrete steps.
2. Estimate the exact number of iterations the agent will likely take to complete the task.
3. Determine which tools will be executed and how the LLM will choose them.
4. Assess the type, complexity, and size of the input/output to the agent at each step.

Use the <scratchpad> tags to think through your reasoning step-by-step before making a final decision. Consider the complexity of the task, the number of subtasks, the amount of information processing required, and the potential challenges the agent might face.

After your analysis, choose the most appropriate difficulty level from the following options:
A. Very easy (suitable for an LLM with 1-2 billion parameters)
B. Easy (suitable for an LLM with 3-4 billion parameters)
C. Moderate (suitable for an LLM with 7-8 billion parameters)
D. Hard (suitable for an LLM with 12-14 billion parameters)
E. Very hard (suitable for an LLM with 30-32 billion parameters)"""

##### Prompt element 5: Examples
# Provide Claude with at least one example of an ideal response that it can emulate. Encase this in <example></example> XML tags. Feel free to provide multiple examples.
# If you do provide multiple examples, give Claude context about what it is an example of, and enclose each example in its own set of XML tags.
# Examples are probably the single most effective tool in knowledge work for getting Claude to behave as desired.
# Make sure to give Claude examples of common edge cases. If your prompt uses a scratchpad, it's effective to give examples of how the scratchpad should look.
# Generally more examples = better.
EXAMPLES = ""

##### Prompt element 6: Input data to process
# If there is data that Claude needs to process within the prompt, include it here within relevant XML tags.
# Feel free to include multiple pieces of data, but be sure to enclose each in its own set of XML tags.
# This element may not be necessary depending on task. Ordering is also flexible.
INPUT_DATA = ""

##### Prompt element 7: Immediate task description or request #####
# "Remind" Claude or tell Claude exactly what it's expected to immediately do to fulfill the prompt's task.
# This is also where you would put in additional variables like the user's question.
# It generally doesn't hurt to reiterate to Claude its immediate task. It's best to do this toward the end of a long prompt.
# This will yield better results than putting this at the beginning.
# It is also generally good practice to put the user's query close to the bottom of the prompt.
IMMEDIATE_TASK = ""

##### Prompt element 8: Precognition (thinking step by step)
# For tasks with multiple steps, it's good to tell Claude to think step by step before giving an answer
# Sometimes, you might have to even say "Before you give your answer..." just to make sure Claude does this first.
# Not necessary with all prompts, though if included, it's best to do this toward the end of a long prompt and right after the final immediate task request or description.
PRECOGNITION = ""

##### Prompt element 9: Output formatting
# If there is a specific way you want Claude's response formatted, clearly tell Claude what that format is.
# This element may not be necessary depending on the task.
# If you include it, putting it toward the end of the prompt is better than at the beginning.
OUTPUT_FORMATTING = """Provide your reasoning and final answer in the following format:
<scratchpad>
[Your step-by-step reasoning here]
</scratchpad>

<answer>[A single capitalized letter for the selected difficulty level]</answer>"""

##### Prompt element 10: Prefilling Claude's response (if any)
# A space to start off Claude's answer with some prefilled words to steer Claude's behavior or response.
# If you want to prefill Claude's response, you must put this in the `assistant` role in the API call.
# This element may not be necessary depending on the task.
PREFILL = ""

######################################## COMBINE ELEMENTS ########################################

PROMPT = ""

if TASK_CONTEXT:
    PROMPT += f"""{TASK_CONTEXT}"""

if TONE_CONTEXT:
    PROMPT += f"""\n\n{TONE_CONTEXT}"""

if TASK_DESCRIPTION:
    PROMPT += f"""\n\n{TASK_DESCRIPTION}"""

if EXAMPLES:
    PROMPT += f"""\n\n{EXAMPLES}"""

if INPUT_DATA:
    PROMPT += f"""\n\n{INPUT_DATA}"""

if IMMEDIATE_TASK:
    PROMPT += f"""\n\n{IMMEDIATE_TASK}"""

if PRECOGNITION:
    PROMPT += f"""\n\n{PRECOGNITION}"""

if OUTPUT_FORMATTING:
    PROMPT += f"""\n\n{OUTPUT_FORMATTING}"""

# Print full prompt
print(PROMPT)

You are an AI tasked with assessing the difficulty of a given task for a local AI agent powered by a large language model (LLM). Your goal is to determine the most cost-efficient local LLM size to complete the task. Accurate difficulty estimation is crucial for optimal LLM selection.

First, you will be presented with the task that the agent needs to solve. Read it carefully:

<agent_task>
Find all person names at https://dzungvpham.github.io/cv/cv_2025.pdf
</agent_task>

The agent will alternate between calling the LLMs to reason about the task's current state and executing tools. The agent can use the following tools: web_search, visit_webpage.

To assess the difficulty of the task, follow these steps:
1. Consider how the agent will plan and break down the task into concrete steps.
2. Estimate the exact number of iterations the agent will likely take to complete the task.
3. Determine which tools will be executed and how the LLM will choose them.
4. Assess the type, complexity, and s

In [46]:
# Ollama
raw_result = chat(
    model="qwen3:1.7b",
    messages=[
        {"role": "user", "content": PROMPT + " /no_think"},
        # {"role": "assistant", "content": PREFILL}
    ],
    options={
        "temperature": 0.0,
    },
)
# print("Thinking:", raw_result.message.thinking)
print(raw_result.message.content)

<think>

</think>

<scratchpad>
To determine the difficulty level of the task "What's the name of the women's liberal arts college in Cambridge, Massachusetts?" for an AI agent, we need to analyze the task step-by-step:

1. **Understanding the Task**: The task is to identify the name of a specific institution. It is a factual query about a known entity (a college in Cambridge, MA).

2. **Plan and Breakdown**:
   - The agent needs to find the name of a women's liberal arts college in Cambridge, Massachusetts.
   - This requires accessing a reliable source of information (e.g., a database, website, or search engine).

3. **Tools to Use**:
   - The agent can use the `web_search` tool to look up information about colleges in Cambridge, MA.
   - The `visit_webpage` tool could be used to directly access a specific webpage that lists colleges in Cambridge.

4. **Estimating Complexity**:
   - The task is straightforward: it involves searching for a specific piece of information.
   - The agent

In [48]:
# Claude
# MODEL_NAME = "claude-sonnet-4-20250514"
MODEL_NAME = "claude-3-5-haiku-20241022"
raw_result = get_claude_completion(MODEL_NAME, PROMPT, temp=0.0)
print(raw_result)
start_match = "<answer>"
end_match = "</answer>"
parsed_result = raw_result[raw_result.find(start_match) + len(start_match):raw_result.rfind(end_match)]
print(parsed_result.strip())

<scratchpad>
Task Analysis:
1. Task Breakdown:
- Requires downloading and extracting names from a PDF document
- Involves web interaction and document parsing
- Needs precise name extraction from a CV/resume

2. Potential Subtasks:
- Use visit_webpage tool to fetch the PDF
- Download/read the PDF content
- Extract person names from the document
- Potentially handle potential parsing challenges

3. Tool Usage:
- visit_webpage will be critical
- Might require additional PDF parsing capabilities
- Name extraction requires semantic understanding

4. Complexity Considerations:
- Single specific URL
- Structured document (CV)
- Relatively constrained extraction task
- Requires precise named entity recognition
- Moderate complexity in document parsing

5. Potential Challenges:
- PDF formatting variations
- Ensuring accurate name extraction
- Handling potential encoding or layout issues
- Requiring some contextual understanding

6. Estimated Iterations:
- Likely 2-3 iterations:
  1. Fetch docu